In [1]:
import torch
import gc
from typing import Optional, Tuple, Any
from transformers import AutoModelForCausalLM, AutoTokenizer

def print_gpu_memory():
    """Prints detailed GPU memory usage"""
    for i in range(torch.cuda.device_count()):
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1024**2
        allocated = torch.cuda.memory_allocated(i) / 1024**2
        reserved = torch.cuda.memory_reserved(i) / 1024**2
        free = total_memory - reserved
        print(f"GPU {i} Memory:")
        print(f"- Total: {total_memory:.0f}MB")
        print(f"- Reserved by cache: {reserved:.0f}MB")
        print(f"- Allocated: {allocated:.0f}MB")
        print(f"- Free: {free:.0f}MB")

def clear_gpu_memory(device_id: Optional[int] = None):
    """
    Aggressively cleans up GPU memory
    
    Args:
        device_id: If provided, only clear this GPU's memory
    """
    # First collect Python garbage to clean any dereferenced tensors
    gc.collect()
    
    # Validate device_id
    device_count = torch.cuda.device_count()
    if device_id is not None:
        if device_id >= device_count:
            raise ValueError(f"Invalid device_id {device_id}. Only {device_count} devices available.")
        devices = [device_id]
    else:
        devices = range(device_count)
    
    # Try to clear each device
    for device in devices:
        try:
            # First try to clear CUDA cache
            torch.cuda.empty_cache()
            
            # Then try device-specific operations
            with torch.cuda.device(f'cuda:{device}'):
                torch.cuda.empty_cache()
                if hasattr(torch.cuda.memory, 'empty_cache'):
                    torch.cuda.memory.empty_cache()
                torch.cuda.synchronize()
        except Exception as e:
            print(f"Warning: Error clearing GPU {device}: {str(e)}")
            
    # Delete all objects still in memory that have .cuda()
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj):
                if obj.is_cuda:
                    del obj
        except Exception:
            pass
    
    # One final garbage collection
    gc.collect()
    torch.cuda.empty_cache()

In [2]:
# Print memory status before cleanup
print("Before cleanup:")
print_gpu_memory()

# Clear GPU 0 specifically
print(" Clearing GPU 0 memory...")
clear_gpu_memory(0)  # Changed from 1 to 0

# Print memory status after cleanup
print("\nAfter cleanup:")
print_gpu_memory()

Before cleanup:
GPU 0 Memory:
- Total: 24260MB
- Reserved by cache: 0MB
- Allocated: 0MB
- Free: 24260MB
 Clearing GPU 0 memory...


/home/guests/andreea_magureanu/.local/lib/python3.11/site-packages/torch/__init__.py:1125: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)



After cleanup:
GPU 0 Memory:
- Total: 24260MB
- Reserved by cache: 0MB
- Allocated: 0MB
- Free: 24260MB


In [3]:
def _cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
_cleanup()

In [3]:
# Check what processes are using GPU memory
import subprocess
import torch

def get_gpu_processes():
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory', '--format=csv,noheader'],
            encoding='utf-8'
        )
        print("=== GPU Processes ===")
        print("PID | Process Name | Memory Used")
        print("-" * 40)
        for line in result.strip().split('\n'):
            if line.strip():
                print(line)
    except Exception as e:
        print(f"Error getting GPU processes: {e}")

get_gpu_processes()

# Also check CUDA memory details
print("\n=== CUDA Memory Details ===")
torch.cuda.empty_cache()  # Try to clear cache first
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:")
    print(f"Memory allocated: {torch.cuda.memory_allocated(i) / 1024**2:.1f} MB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(i) / 1024**2:.1f} MB")
    print(f"Max memory allocated: {torch.cuda.max_memory_allocated(i) / 1024**2:.1f} MB")

=== GPU Processes ===
PID | Process Name | Memory Used
----------------------------------------
840846, python, 5070 MiB
869051, /home/guests/andreea_magureanu/.conda/envs/rare_dis/bin/python, 8802 MiB
878018, /home/guests/andreea_magureanu/.conda/envs/rare_dis/bin/python, 254 MiB

=== CUDA Memory Details ===
  GPU 0:
Memory allocated: 0.0 MB
Memory reserved: 0.0 MB
Max memory allocated: 0.0 MB


In [5]:
import torch
import gc

def real_cleanup():
    # 1. Delete all references to GPU tensors and models
    global model, optimizer, data  # Reference your variables here
    
    if 'model' in globals():
        del model
    if 'optimizer' in globals():
        del optimizer
    if 'data' in globals():
        del data
    
    # 2. Clear any other GPU tensors
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                del obj
        except:
            pass
    
    # 3. Force garbage collection
    gc.collect()
    
    # 4. Clear CUDA cache (now actually effective)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Usage:
real_cleanup()

/home/guests/andreea_magureanu/.local/lib/python3.11/site-packages/torch/__init__.py:1125: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [6]:
if 'model' in globals():
       print("MODEL")
if 'optimizer' in globals():
        print("OPTIMI")
if 'data' in globals():
        print("DTATA")

In [4]:
def _cuda_free_gib(device=0) -> float:
    if not torch.cuda.is_available(): 
        return 0.0
    free, total = torch.cuda.mem_get_info(device)
    return free / (1024**3)

free_gib = _cuda_free_gib()
print(free_gib)

23.43438720703125


In [5]:
print(torch.cuda.mem_get_info(0))

(12316835840, 12642746368)


In [5]:
import os
import signal

# Kill the processes using GPU memory
os.kill(840846, signal.SIGKILL)  # Kill the main process
#os.kill(1167917, signal.SIGKILL)  # Kill the secondary process

PermissionError: [Errno 1] Operation not permitted

In [6]:
import requests
from lxml import etree as ET

PMCID = "PMC9314610"

URLS = [
    f"https://www.ebi.ac.uk/europepmc/webservices/rest/{PMCID}/fullTextXML",
    f"https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi?verb=GetRecord&identifier=oai:pubmedcentral.nih.gov:{PMCID}&metadataPrefix=pmc",
]

DROP_HEADS = {
    "acknowledgements","acknowledgments","funding","funding information",
    "conflict of interest","competing interests","author contributions",
    "data availability","ethics","references","bibliography",
    "supplementary","appendix","correspondence"
}

def fetch_xml():
    last_err = None
    for url in URLS:
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            # Quick sanity check: must contain an <article> element
            if b"<article" in r.content:
                return ET.fromstring(r.content)
            # Some wrappers (OAI) put <article> deeper; still okay
            try:
                root = ET.fromstring(r.content)
                if root.xpath("//*[local-name()='article']"):
                    return root
            except ET.XMLSyntaxError as e:
                last_err = e
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to fetch XML for {PMCID}: {last_err}")

def norm_text(node):
    return " ".join(" ".join(node.itertext()).split())

def parse_title_abs_paras(root):
    # Find the article node regardless of namespaces/wrappers
    article_nodes = root.xpath("//*[local-name()='article']")
    if not article_nodes:
        return "", "", []
    article = article_nodes[0]

    # Title
    title_nodes = article.xpath(".//*[local-name()='article-title']")
    title = norm_text(title_nodes[0]) if title_nodes else ""

    # Abstract (join all parts)
    abs_nodes = article.xpath(".//*[local-name()='abstract']")
    abstract = norm_text(abs_nodes[0]) if abs_nodes else ""

    # Body → sections → paragraphs (skip unwanted heads)
    paras = []
    for sec in article.xpath(".//*[local-name()='body']//*[local-name()='sec']"):
        head = " ".join(sec.xpath(".//*[local-name()='title']/text()")).strip().lower()
        if any(bad in head for bad in DROP_HEADS):
            continue
        for p in sec.xpath(".//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    # Fallback: any <p> under body if secs were missed
    if not paras:
        for p in article.xpath(".//*[local-name()='body']//*[local-name()='p']"):
            t = norm_text(p)
            if len(t.split()) >= 5:
                paras.append(t)

    return title, abstract, paras

if __name__ == "__main__":
    root = fetch_xml()
    title, abstract, paras = parse_title_abs_paras(root)

    print("\nTITLE:\n", title, "\n")
    print("ABSTRACT (first 500 chars):\n", abstract[:500], "...\n")
    print("FIRST 3 PARAGRAPHS:\n")
    for i, p in enumerate(paras[:3], 1):
        print(f"[{i}] {p}\n")
    print(f"Total paragraphs found: {len(paras)}")



TITLE:
 Genotype–phenotype correlates in Joubert syndrome: A review 

ABSTRACT (first 500 chars):
 Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic varian ...

FIRST 3 PARAGRAPHS:

[1] Joubert syndrome (JS) is a rare congenital neurodevelopmental primary ciliopathy with a population‐based prevalence reaching 1.7 per 100,000 in the age range 0–19 years (Nuovo et al., 2020 ). First described by Dr Marie Joubert about 50 years ago (Joubert, Eisenring, Robb, & Andermann, 1969 ), JS is now diagnosed upon recognition of a pathognomonic malformation of th

In [45]:
print(abstract)

Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic variants in TMEM67 have a significantly higher risk of liver fibrosis, while pathogenic variants in NPHP1 , RPGRIP1L , and TMEM237 are frequently associated to JS with renal involvement, requiring a closer monitoring of liver parameters, or renal functioning. On the other hand, individuals with causal variants in the CEP290 or AHI1 need a closer surveillance for retinal dystrophy and, in case of CEP290 , also for chronic kidney disease. These examples highlight how an accurate description of the range

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"

# pick the best dtype for your GPU (bfloat16 if supported, else float16)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# load tokenizer from disk; use_fast uses the Rust tokenizer for speed:contentReference[oaicite:2]{index=2}
tok = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,     # don't download anything, use local files:contentReference[oaicite:3]{index=3}
    use_fast=True,
    trust_remote_code=True,    # allows loading custom model code:contentReference[oaicite:4]{index=4}
)

# ensure a padding token exists (decoder-only models often reuse the EOS token for padding)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

# load the model weights in the selected dtype
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=True,
    torch_dtype=dtype,         # load in float16 or bfloat16:contentReference[oaicite:5]{index=5}
).to("cuda")

print("Loaded model ok – using dtype:", dtype)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded model ok – using dtype: torch.bfloat16


In [7]:
import gc, os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

MODEL_DIR = "/home/guests/andreea_magureanu/projects/rare_disease/models/medgemma-4b-it"

def _cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def _is_oom(e: Exception) -> bool:
    m = str(e).lower()
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in m or "cuda oom" in m


tok = AutoTokenizer.from_pretrained(
    MODEL_DIR, local_files_only=True, use_fast=True, trust_remote_code=True
)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

try:
    _cleanup()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        local_files_only=True,
        trust_remote_code=True,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    ).to("cuda")
    mode = f"fp{16 if dtype==torch.float16 else 'bf16'}"
    print("Loaded model ok - using dtype:", dtype)

except Exception as e:
    if not _is_oom(e):
        raise  

    print("OOM on fp16/bf16 load. Falling back to bitsandbytes 4-bit…")
    _cleanup()

    try:
       
        bnb4 = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16),
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            trust_remote_code=True,
            local_files_only=True,
            device_map="auto",         
            quantization_config=bnb4,
            low_cpu_mem_usage=True,
        )
        mode = "4bit_nf4_auto"
        print("Loaded with bitsandbytes 4-bit (NF4).")

    except Exception as e4:
        if _is_oom(e4):
            print("Still OOM on 4-bit. Trying 8-bit…")
        else:
            print(f"4-bit failed ({type(e4).__name__}: {e4}). Trying 8-bit…")
        _cleanup()
        try:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_DIR,
                trust_remote_code=True,
                local_files_only=True,
                device_map="auto",
                load_in_8bit=True,       
                low_cpu_mem_usage=True,
            )
            mode = "8bit_auto"
            print("Loaded with bitsandbytes 8-bit.")
        except Exception as e8:
          
            print(f"8-bit failed ({type(e8).__name__}: {e8}). Trying auto offload…")
            _cleanup()
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_DIR,
                trust_remote_code=True,
                local_files_only=True,
                device_map="auto",
                max_memory={0: "90%", "cpu": "64GiB"},  # avoid crash, spill to CPU
                torch_dtype="auto",
                low_cpu_mem_usage=True,
            )
            mode = "half_auto_offload"
            print("Loaded with auto offload (GPU+CPU).")

print("Mode:", mode)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded model ok - using dtype: torch.bfloat16
Mode: fpbf16


In [8]:
import re, json
def chat(messages, max_new_tokens=800):
    device = "cuda"
    prompt_text = tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # mark where assistant should start
    )

    enc = tok(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    enc = {k: v.to(device) for k, v in enc.items()}


    out_ids = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,     # start deterministic
               # <- be explicit
        top_p=1.0,
        repetition_penalty=1.0,
        eos_token_id=tok.eos_token_id,   # clean stop
        pad_token_id=tok.pad_token_id,  
        use_cache=True,# silence warnings
        )

    # Decode only the new tokens (no prompt echo)
    new_tokens = out_ids[0, enc["input_ids"].shape[-1]:]
    out_text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return out_text

Step 1

In [32]:
#DOESTN EXTRACT Causal genes
system_entity_extraction = (
    "You are a precise medical named entity recognition (NER) agent. "
    "Your task is to identify and list medical entities from the provided text. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of strings. "
    "List the primary, canonical name for each unique entity related to: "
    "'rare_disease', 'phenotype', 'causal_gene', 'genotype', 'treatment'. "
    "If an entity is referred to by multiple names (e.g., 'SCN1A' and 'sodium voltage-gated channel alpha subunit 1'), "
    "choose the most standard or full name as it first appears as the canonical name for this list. "
    "Do not output any other keys or explanatory text."
)
user_entity_extraction = (
    f"Title: {title}\nAbstract: {abstract}\n\n"
    "Extract the canonical names of all entities related to rare diseases, phenotypes, causal genes, genotypes, and treatments. "
    "Output a simple list of entity names as a JSON object."
)
messages_entity = [
    {"role": "system", "content": system_entity_extraction},
    {"role": "user",   "content": user_entity_extraction},
]


In [9]:
# Best VARIANT FOR ENTITIES!!!!!

system_step1 = (
    "You are a precise medical information extraction agent. Your task is to identify and normalize medical entities from the provided text. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each entity object must have: "
    "'name' (string, the primary/canonical name as used in the text), "
    "'type' (string, must be one of: 'rare_disease', 'phenotype', 'gene', 'genotype', 'treatment'), "
   "If an entity is mentioned multiple times, keep just the first occurrence "
    "Do not output any relations. Do not output any other keys or explanatory text."
)
user_step1 = (
    f"Title: {title}\n"
    f"Abstract: {abstract}\n\n"
    "Task: Extract entities of types rare_disease, phenotype, gene, genotype, treatment, "
    "with explicit aliases only. Output the JSON now."
)

messages_entity = [
    {"role": "system", "content": system_step1},
    {"role": "user",   "content": user_step1},
]

In [10]:
entities = chat(messages_entity)
print(entities)

```json
{
  "entities": [
    {
      "name": "Joubert syndrome",
      "type": "rare_disease"
    },
    {
      "name": "cerebellar and brainstem malformation",
      "type": "phenotype"
    },
    {
      "name": "molar tooth sign",
      "type": "phenotype"
    },
    {
      "name": "liver fibrosis",
      "type": "phenotype"
    },
    {
      "name": "renal involvement",
      "type": "phenotype"
    },
    {
      "name": "retinal dystrophy",
      "type": "phenotype"
    },
    {
      "name": "chronic kidney disease",
      "type": "phenotype"
    },
    {
      "name": "TMEM67",
      "type": "gene"
    },
    {
      "name": "NPHP1",
      "type": "gene"
    },
    {
      "name": "RPGRIP1L",
      "type": "gene"
    },
    {
      "name": "TMEM237",
      "type": "gene"
    },
    {
      "name": "CEP290",
      "type": "gene"
    },
    {
      "name": "AHI1",
      "type": "gene"
    }
  ]
}
```


In [12]:
def extract_entity_names(text: str) -> list:
    """
    Extract just the entity names from the model output, even if JSON is malformed.
    Returns a list of entity names.
    """
    names = set()
    
    # Try to find any JSON-like structure first
    json_matches = re.finditer(r'\{[^{}]*\}', text)
    
    for match in json_matches:
        try:
            data = json.loads(match.group(0))
            if 'entities' in data and isinstance(data['entities'], list):
                for entity in data['entities']:
                    if isinstance(entity, dict) and 'name' in entity:
                        names.add(entity['name'])
        except json.JSONDecodeError:
            continue
    
    # If no JSON found, try to extract names directly using regex patterns
    if not names:
        # Look for name patterns like "name": "something"
        name_matches = re.finditer(r'"name"\s*:\s*"([^"]+)"', text)
        for match in name_matches:
            names.add(match.group(1))
    
    return names

# Test with your last output
print("Extracting entity names from output:")
entity_names = extract_entity_names(entities)
print("\nFound entity names:")
for name in entity_names:
    print(f"- {name}")

# These names can now be used in your next prompt for alias extraction

Extracting entity names from output:

Found entity names:
- chronic kidney disease
- NPHP1
- renal involvement
- RPGRIP1L
- CEP290
- retinal dystrophy
- molar tooth sign
- AHI1
- TMEM67
- cerebellar and brainstem malformation
- Joubert syndrome
- TMEM237
- liver fibrosis


In [63]:
print(entities)

```json
{
  "entities": [
    {
      "name": "Joubert syndrome",
      "type": "rare_disease"
    },
    {
      "name": "cerebellar and brainstem malformation",
      "type": "phenotype"
    },
    {
      "name": "molar tooth sign",
      "type": "phenotype"
    },
    {
      "name": "liver fibrosis",
      "type": "phenotype"
    },
    {
      "name": "renal involvement",
      "type": "phenotype"
    },
    {
      "name": "retinal dystrophy",
      "type": "phenotype"
    },
    {
      "name": "chronic kidney disease",
      "type": "phenotype"
    },
    {
      "name": "TMEM67",
      "type": "gene"
    },
    {
      "name": "NPHP1",
      "type": "gene"
    },
    {
      "name": "RPGRIP1L",
      "type": "gene"
    },
    {
      "name": "TMEM237",
      "type": "gene"
    },
    {
      "name": "CEP290",
      "type": "gene"
    },
    {
      "name": "AHI1",
      "type": "gene"
    }
  ]
}
```


In [13]:
print(entity_names)
print(type(entity_names))
import json 
# Convert set to list before JSON serialization
entity_names_json = json.dumps(list(entity_names), ensure_ascii=False)
print(entity_names_json)

{'TMEM67', 'RPGRIP1L', 'AHI1', 'Joubert syndrome', 'liver fibrosis', 'cerebellar and brainstem malformation', 'renal involvement', 'molar tooth sign', 'retinal dystrophy', 'chronic kidney disease', 'CEP290', 'NPHP1', 'TMEM237'}
<class 'set'>
["TMEM67", "RPGRIP1L", "AHI1", "Joubert syndrome", "liver fibrosis", "cerebellar and brainstem malformation", "renal involvement", "molar tooth sign", "retinal dystrophy", "chronic kidney disease", "CEP290", "NPHP1", "TMEM237"]


In [15]:
# LETS WORK WITH THIS TILL NOW ALIASES
# testing 1
system_alias_resolution = (
    "You are a meticulous alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "Your task is to find ALL explicitly stated aliases for each entity FROM THE TEXT. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each entity object must have: "
    "'name' (string, the primary/canonical name as used in the text), "
    "'aliases' (array of strings) which may be empty if not present in the text"
    "CRITICAL RULES: "
    " - An alias MUST be explicitly linked in the text (e.g., in parentheses: 'Duchenne muscular dystrophy (DMD)', "
    "   or after phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    " - DO NOT invent any aliases. If no alias exists for an entity, assign it an empty array []. "
    " - Include common abbreviations only if they are explicitly presented as an alternative name. "
    "Do not output any other keys or explanatory text."
)
user_alias_resolution = (
    f"Text: {title} {abstract}\n\n"
    "Canonical Entity List: {entities}\n\n" 
    "For each entity in the list, find all explicitly stated aliases from the text. If none found, use an empty array. "
    "Output a JSON object mapping each entity to its aliases."
)
"""
```json
{
  "entities": [
    {
      "name": "Joubert syndrome",
      "aliases": [
        "JS"
      ]
    },
    {
      "name": "TMEM67",
      "aliases": []
    },
    {
      "name": "NPHP1",
      "aliases": []
    },
    {
      "name": "RPGRIP1L",
      "aliases": []
    },
    {
      "name": "TMEM237",
      "aliases": []
    },
    {
      "name": "CEP290",
      "aliases": []
    },
    {
      "name": "AHI1",
      "aliases": []
    }
  ]
}
```
"""

'\n```json\n{\n  "entities": [\n    {\n      "name": "Joubert syndrome",\n      "aliases": [\n        "JS"\n      ]\n    },\n    {\n      "name": "TMEM67",\n      "aliases": []\n    },\n    {\n      "name": "NPHP1",\n      "aliases": []\n    },\n    {\n      "name": "RPGRIP1L",\n      "aliases": []\n    },\n    {\n      "name": "TMEM237",\n      "aliases": []\n    },\n    {\n      "name": "CEP290",\n      "aliases": []\n    },\n    {\n      "name": "AHI1",\n      "aliases": []\n    }\n  ]\n}\n```\n'

In [31]:
# LETS WORK WITH THIS TILL NOW ALIASES
# testing 2
system_alias_resolution = (
    "You are a meticulous alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "Your task is to CHECK for all canonical entities GIVEN, if there are ANY explicitly stated aliases for each entity FROM THE TEXT. "
    "Output MUST be a SINGLE JSON object with a list of objects. "
    "Each object must have: "
    "'name' (string, the primary/canonical name GIVEN), "
    "'aliases' (array of strings) which may be empty if not present in the text"
    "CRITICAL RULES: "
    " - An alias MUST be explicitly linked in the text (e.g., in parentheses: 'Duchenne muscular dystrophy (DMD)', "
    "   or after phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    " - DO NOT invent any aliases. If no alias exists for an entity, output an empty array. "
    " - Include common abbreviations only if they are explicitly presented as an alternative name. "
    "Do not output any other keys or explanatory text."
)
user_alias_resolution = (
    f"Text: {title} {abstract}\n\n"
    "Canonical Entity List: {entity_names}\n\n" 
    "For each entity in the list, check if there  explicitly stated aliases in the text. If none found, output an empty array. "
    "Output a JSON object ."
)


In [17]:
# testing 3

system_alias_resolution = (
    "You are a meticulous alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "Your task is to return ALL of these entities, each with any explicitly stated aliases found in the text. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each object MUST correspond to ONE entity from the input list, no matter if aliases are found. "
    "Fields required per object: "
    " - 'name': the canonical name (exactly as provided in the input list). "
    " - 'aliases': an array of strings. Use [] if none are found. "
    "CRITICAL RULES: "
    " - You MUST include every input entity exactly once in the output, even if 'aliases' is empty. "
    " - An alias MUST be explicitly linked in the text (e.g., parentheses: 'Duchenne muscular dystrophy (DMD)', "
    "   or phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    " - DO NOT invent aliases. "
    " - Do not output any other keys or text besides the JSON object."
)
user_alias_resolution = (
    f"Text:\n{title} {abstract}\n\n"
    f"Canonical Entity List:\n{entities}\n\n"
    "Instruction: For EACH entity in the Canonical Entity List, return an object with its 'name' "
    "and any explicitly stated 'aliases' found in the text. "
    "If no aliases are present for that entity, still include it with 'aliases': []."
)


In [26]:
# testing 4 
system_alias_resolution = (
    "You are a meticulous alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "For EACH entity given output ALL explicitly stated aliases FROM THE TEXT. If none are found, return an empty array. "
    # "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    # "Each entity object must have: "
    # "'name' (string, the primary/canonical name as used in the text), "
    # "'aliases' (array of strings) which may be empty if not present in the text"
    # "CRITICAL RULES: "
    # " - An alias MUST be explicitly linked in the text (e.g., in parentheses: 'Duchenne muscular dystrophy (DMD)', "
    # "   or after phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    # " - DO NOT invent any aliases. If no alias exists for an entity, assign it an empty array []. "
    # " - Include common abbreviations only if they are explicitly presented as an alternative name. "
    # "Output all given entities, even if no aliases are found. "
    # "Do not output any other keys or explanatory text."
)
user_alias_resolution = (
    f"Text: {title} {abstract}\n\n"
    "Canonical Entity List: {entities}\n\n" 
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each entity object must have: "
    "'name' (string, the primary/canonical name as used in the text), "
    "'aliases' (array of strings) which may be empty if not present in the text"
    "CRITICAL RULES: "
    " - An alias MUST be explicitly linked in the text (e.g., in parentheses: 'Duchenne muscular dystrophy (DMD)', "
    "   or after phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    " - DO NOT invent any aliases. If no alias exists for an entity, assign it an empty array []. "
    " - Include common abbreviations only if they are explicitly presented as an alternative name. "
    "Output all given entities, even if no aliases are found. "
)

In [ ]:
system_alias_resolution = (
    "You are a meticulous alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "Your task is to find ALL explicitly stated aliases for each entity FROM THE TEXT. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each entity object must have: "
    "'name' (string, the primary/canonical name as used in the text), "
    "'aliases' (array of strings) which may be empty if not present in the text. "
    "CRITICAL RULES: "
    " - You MUST process EVERY entity in the provided list. Do not skip any entities."
    " - An alias MUST be explicitly linked in the text (e.g., in parentheses: 'Duchenne muscular dystrophy (DMD)', "
    "   or after phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    " - DO NOT invent any aliases. If no alias exists for an entity, assign it an empty array []. "
    " - Include common abbreviations only if they are explicitly presented as an alternative name. "
    " - The output MUST include exactly the same entities as provided in the input list."
    "Do not output any other keys or explanatory text."
)


entity_list = ["TMEM67", "RPGRIP1L", "AHI1", "Joubert syndrome", "liver fibrosis", 
               "cerebellar and brainstem malformation", "renal involvement", 
               "molar tooth sign", "retinal dystrophy", "chronic kidney disease", 
               "CEP290", "NPHP1", "TMEM237"]

user_alias_resolution = (
    f"Text: {title} {abstract}\n\n"
    "Canonical Entity List (you MUST include ALL of these):\n" + 
    "\n".join([f"- {entity}" for entity in entity_list]) + "\n\n"
    "For each entity in the list above, find all explicitly stated aliases from the text. "
    "If no alias is found, use an empty array. "
    "Output a JSON object with an 'entities' key containing a list of objects for ALL entities."
)

In [52]:
system_alias_resolution = (
    "You are an alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "Your task is to return ONLY one JSON object with the structure: "
    "{\"entities\":[{\"name\": str, \"aliases\": [str]}...]} "
    "Instructions: "
    " - For EACH entity in the provided list, always include exactly one object in the output. "
    " - 'name' must match the canonical name exactly as given in the input. "
    " - 'aliases' must contain ONLY explicit aliases found in the text. "
    " - Aliases are valid ONLY if shown in one of these patterns: "
    "     • Canonical (ALIAS) "
    "     • ALIAS (Canonical) "
    "     • introduced by 'aka', 'also known as', 'abbreviated as', 'short for', or 'formerly called'. "
    " - If no alias is found, use an empty array []. "
    " - Do NOT invent aliases or skip entities. "
    " - Output ONLY the JSON object, no explanations or extra text."
)

user_alias_resolution = (
    f"Text:\n{title} {abstract}\n\n"
    f" Entity List:\n{entities}\n\n"
)


In [65]:
print(entity_names)
print(type(entity_names))

{'renal involvement', 'chronic kidney disease', 'CEP290', 'NPHP1', 'cerebellar and brainstem malformation', 'liver fibrosis', 'retinal dystrophy', 'AHI1', 'TMEM67', 'TMEM237', 'RPGRIP1L', 'Joubert syndrome', 'molar tooth sign'}
<class 'set'>


In [70]:
import json
import re 

def parse_json(output: str):
    """
    Extract the first valid JSON object from model output and return as Python dict.
    Raises ValueError if no JSON found.
    """
    # Sometimes the model adds stray text — extract between first { and last }
    match = re.search(r"\{.*\}", output, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in output")
    
    json_str = match.group(0)
    return json.loads(json_str)

system_alias_resolution = (
    "You are a meticulous alias resolution agent. "
    "You are given a text and a list of canonical entity names. "
    "Your task is to find ALL explicitly stated aliases for each entity FROM THE TEXT. "
    "Output MUST be a SINGLE JSON object with ONLY an 'entities' key, whose value is a list of objects. "
    "Each entity object must have: "
    "'name' (string, the given canonical entity name), "
    "'aliases' (array of strings) which may be empty if not present in the text"
    "CRITICAL RULES: "
    " - An alias MUST be explicitly linked in the text (e.g., in parentheses: 'Duchenne muscular dystrophy (DMD)', "
    "   or after phrases like 'also known as', 'abbreviated as', 'formerly called'). "
    " - DO NOT invent any aliases. If no alias exists for an entity, assign it an empty array []. "
    " - Include common abbreviations only if they are explicitly presented as an alternative name. "
    "Do not output any other keys or explanatory text."
)

def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

batched = []
i=0
for group in chunk(list(entity_names), 4):  # 4 at a time
    i+=1
    user = (
        "TEXT:\n<<<\n" + abstract + "\n>>>\n\n"
        "ENTITIES (JSON):\n" + json.dumps(group, ensure_ascii=False)
    )
    out = chat([
        {"role":"system","content": system_alias_resolution},
        {"role":"user","content": user},
    ])
    print(i)
    print(out)
    
    # batched.append(parse_json(out))

# # merge preserving original order
# aliases_by_name = {e["name"]: [] for e in entity_names}
# for part in batched:
#     for e in part["entities"]:
#         aliases_by_name[e["name"]].extend(e.get("aliases", []))

# final = {"entities":[{"name": n, "aliases": sorted(set(a))} for n,a in aliases_by_name.items()]}


1
```json
{
  "entities": [
    {
      "name": "renal involvement",
      "aliases": [
        "renal involvement"
      ]
    },
    {
      "name": "chronic kidney disease",
      "aliases": [
        "chronic kidney disease"
      ]
    },
    {
      "name": "CEP290",
      "aliases": [
        "CEP290"
      ]
    },
    {
      "name": "NPHP1",
      "aliases": [
        "NPHP1"
      ]
    }
  ]
}
```
```json
{
  "entities": [
    {
      "name": "renal involvement",
      "aliases": [
        "renal involvement"
      ]
    },
    {
      "name": "chronic kidney disease",
      "aliases": [
        "chronic kidney disease"
      ]
    },
    {
      "name": "CEP290",
      "aliases": [
        "CEP290"
      ]
    },
    {
      "name": "NPHP1",
      "aliases": [
        "NPHP1"
      ]
    }
  ]
}
```
```json
{
  "entities": [
    {
      "name": "renal involvement",
      "aliases": [
        "renal involvement"
      ]
    },
    {
      "name": "chronic kidney disease",
 

In [53]:
messages = [
    {"role": "system", "content": system_alias_resolution},
    {"role": "user",   "content": user_alias_resolution},
]
aliases = chat(messages)
print(aliases)

```json
{
  "entities": [
    {
      "name": "Joubert syndrome",
      "aliases": [
        "JS"
      ]
    },
    {
      "name": "molar tooth sign",
      "aliases": []
    },
    {
      "name": "TMEM67",
      "aliases": [
        "TMEM67"
      ]
    }
  ]
}
```


In [3]:
import json
entity_names_json = json.dumps(list(entity_names), ensure_ascii=False)
print(entity_names_json)

NameError: name 'entity_names' is not defined

In [14]:
import json, textwrap, torch
import langextract as lx
from langextract.providers.transformers import TransformersLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "/home/guests/andreea_magureanu/.cache/huggingface/hub/models--openai--gpt-oss-20b/snapshots/6cee5e81ee83917806bbde320786a8fb61efebee"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True, local_files_only=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True
)

oss20b = TransformersLanguageModel(
    model=model,
    tokenizer=tok,
    generation_kwargs={"temperature": 0.0, "max_new_tokens": 512, "do_sample": False}
)

ModuleNotFoundError: No module named 'langextract.providers.transformers'

In [ ]:
#GEMINI
import langextract as lx
import textwrap




prompt = textwrap.dedent(f"""\
    Extract alias relationships for the given biomedical entities.
    Focus on instances where a canonical entity is mentioned with an alternate name 
    (e.g., "also known as ___" or "previously called ___", "aka", "canonical name (aliase)").
    The known canonical entities include: {", ".join(entity_names_json)}.
    For each alias found, use the exact text of the alias and pair it with the canonical name.
    Do not infer aliases that are not explicitly stated in the text.
""")

examples = [
    lx.data.ExampleData(
        text="Joubert syndrome (previously called vermian aplasia) is a rare genetic condition.",
        extractions=[
            lx.data.Extraction(
                extraction_class="alias",
                extraction_text="vermian aplasia",
                attributes={"canonical": "Joubert syndrome"}
            )
        ]
    ),
    lx.data.ExampleData(
        text="Mutations in the CEP290 gene (also known as NPHP6) are a common cause of Joubert syndrome.",
        extractions=[
            lx.data.Extraction(
                extraction_class="alias",
                extraction_text="NPHP6",
                attributes={"canonical": "CEP290"}
            )
        ]
    )
]

result = lx.extract(
    text_or_documents=abstract,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",                    
  
    fence_output=True, 
    use_schema_constraints=False
)



In [ ]:
#OSS MODEL
import langextract as lx
import textwrap




prompt = textwrap.dedent(f"""\
    Extract alias relationships for the given biomedical entities.
    Focus on instances where a canonical entity is mentioned with an alternate name 
    (e.g., "also known as ___" or "previously called ___", "aka", "canonical name (aliase)").
    The known canonical entities include: {", ".join(entity_names_json)}.
    For each alias found, use the exact text of the alias and pair it with the canonical name.
    Do not infer aliases that are not explicitly stated in the text.
""")

examples = [
    lx.data.ExampleData(
        text="Joubert syndrome (previously called vermian aplasia) is a rare genetic condition.",
        extractions=[
            lx.data.Extraction(
                extraction_class="alias",
                extraction_text="vermian aplasia",
                attributes={"canonical": "Joubert syndrome"}
            )
        ]
    ),
    lx.data.ExampleData(
        text="Mutations in the CEP290 gene (also known as NPHP6) are a common cause of Joubert syndrome.",
        extractions=[
            lx.data.Extraction(
                extraction_class="alias",
                extraction_text="NPHP6",
                attributes={"canonical": "CEP290"}
            )
        ]
    )
]

result = lx.extract(
    text_or_documents=abstract,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",                    
   
    fence_output=True, 
    use_schema_constraints=False
)



NameError: name 'entity_names_json' is not defined

In [16]:

aliases = []
for ext in result.extractions: 
    if ext.extraction_class == "alias":
        canonical = ext.attributes.get("canonical")
        alias_name = ext.extraction_text
        aliases.append((canonical, alias_name))
        print(f"Alias found: {canonical} -> {alias_name}")




Alias found: Joubert syndrome -> JS


In [15]:
print(result)

AnnotatedDocument(extractions=[Extraction(extraction_class='alias', extraction_text='JS', char_interval=CharInterval(start_pos=27, end_pos=29), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=1, group_index=0, description=None, attributes={'canonical': 'Joubert syndrome'}), Extraction(extraction_class='alias', extraction_text='molar tooth sign', char_interval=CharInterval(start_pos=158, end_pos=174), alignment_status=<AlignmentStatus.MATCH_EXACT: 'match_exact'>, extraction_index=2, group_index=1, description=None, attributes={'canonical': 'cerebellar and brainstem malformation'})], text='Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of g

In [17]:
from collections import defaultdict

entities_dict = defaultdict(set)

for ext in result.extractions: 
    entities_dict[ext.attributes.get("canonical")].add(ext.extraction_text)
    
print(dict(entities_dict))

{'Joubert syndrome': {'JS'}}


In [18]:
for name in entity_names:
    entities_dict[name]  # ensure every entity is present

In [19]:
print(entities_dict)

defaultdict(<class 'set'>, {'Joubert syndrome': {'JS'}, 'chronic kidney disease': set(), 'NPHP1': set(), 'renal involvement': set(), 'RPGRIP1L': set(), 'CEP290': set(), 'retinal dystrophy': set(), 'molar tooth sign': set(), 'AHI1': set(), 'TMEM67': set(), 'cerebellar and brainstem malformation': set(), 'TMEM237': set(), 'liver fibrosis': set()})


In [26]:
def to_entity_list(name_to_aliases):
    out = []
    for i, (name, aliases) in enumerate(name_to_aliases.items(), start=1):
        out.append({"id": f"E{i}", "name": name, "aliases": sorted(list(aliases))})
    return out

entities_json = json.dumps(to_entity_list(entities_dict))
print(entities_json)

[{"id": "E1", "name": "Joubert syndrome", "aliases": ["JS"]}, {"id": "E2", "name": "chronic kidney disease", "aliases": []}, {"id": "E3", "name": "NPHP1", "aliases": []}, {"id": "E4", "name": "renal involvement", "aliases": []}, {"id": "E5", "name": "RPGRIP1L", "aliases": []}, {"id": "E6", "name": "CEP290", "aliases": []}, {"id": "E7", "name": "retinal dystrophy", "aliases": []}, {"id": "E8", "name": "molar tooth sign", "aliases": []}, {"id": "E9", "name": "AHI1", "aliases": []}, {"id": "E10", "name": "TMEM67", "aliases": []}, {"id": "E11", "name": "cerebellar and brainstem malformation", "aliases": []}, {"id": "E12", "name": "TMEM237", "aliases": []}, {"id": "E13", "name": "liver fibrosis", "aliases": []}]


In [45]:
system_stepREL = (
    "You are a precise biomedical RELATION extraction agent. "
    "Your task is to extract relations ONLY between the provided entities. "
    "Aliases MUST be treated as the SAME entity as their canonical name. "
    "Alias matching rules: case-insensitive exact match on canonical name OR any alias; "
    "allow trivial variants (plural 's', hyphen↔space, parentheses). "
    "Do NOT invent entities beyond those provided. "
    "If a mention could match multiple entities, SKIP it (do not guess). "
    "Output MUST be a SINGLE JSON object with ONLY a 'relations' key whose value is a list of objects. "
    "Each relation object MUST have: "
    "'head' (string; canonical name), "
    "'tail' (string; canonical name), "
    "'relation' (string; )"#one of: 'caused_by','associated_with','presents_with','treated_with'), "
    # "'evidence' (string; the SHORTEST supporting sentence from the text), "
    # "'confidence' (number in [0,1]). "
    "Do not output any other keys or explanatory text."
)


user_stepREL = (
    f"Entities (canonical + aliases):\n"
    f"{entities_json}\n\n"
    # "Allowed relation types:\n"
    # "['caused_by','associated_with','presents_with','treated_with']\n\n"
    f"Text:\n{abstract}\n\n"
    "Task: Extract ONLY relations between the provided entities, mapping any alias mentions "
    "to their canonical names as listed. Output the JSON now."
)


In [39]:
print(abstract)
print(entities_json)

Abstract Joubert syndrome (JS) is a genetically heterogeneous primary ciliopathy characterized by a pathognomonic cerebellar and brainstem malformation, the “molar tooth sign,” and variable organ involvement. Over 40 causative genes have been identified to date, explaining up to 94% of cases. To date, gene‐phenotype correlates have been delineated only for a handful of genes, directly translating into improved counseling and clinical care. For instance, JS individuals harboring pathogenic variants in TMEM67 have a significantly higher risk of liver fibrosis, while pathogenic variants in NPHP1 , RPGRIP1L , and TMEM237 are frequently associated to JS with renal involvement, requiring a closer monitoring of liver parameters, or renal functioning. On the other hand, individuals with causal variants in the CEP290 or AHI1 need a closer surveillance for retinal dystrophy and, in case of CEP290 , also for chronic kidney disease. These examples highlight how an accurate description of the range

In [48]:
messages_relations = [
    {"role": "system", "content": system_stepREL},
    {"role": "user",   "content": user_stepREL},
]

out = chat(messages_relations, max_new_tokens=1000)

In [49]:
print(out)

```json
{
  "relations": [
    {
      "head": "Joubert syndrome",
      "tail": "JS",
      "relation": "is_a"
    },
    {
      "head": "Joubert syndrome",
      "tail": "cerebellar and brainstem malformation",
      "relation": "has"
    },
    {
      "head": "Joubert syndrome",
      "tail": "molar tooth sign",
      "relation": "has"
    },
    {
      "head": "Joubert syndrome",
      "tail": "renal involvement",
      "relation": "has"
    },
    {
      "head": "Joubert syndrome",
      "tail": "liver fibrosis",
      "relation": "has"
    },
    {
      "head": "Joubert syndrome",
      "tail": "chronic kidney disease",
      "relation": "has"
    },
    {
      "head": "Joubert syndrome",
      "tail": "NPHP1",
      "relation": "associated_with"
    },
    {
      "head": "Joubert syndrome",
      "tail": "RPGRIP1L",
      "relation": "associated_with"
    },
    {
      "head": "Joubert syndrome",
      "tail": "TMEM237",
      "relation": "associated_with"
    },
    {
 